In [1]:
import sys
print(sys.executable)
!{sys.executable} -m pip install scikit-learn pandas numpy scipy xgboost


/opt/homebrew/opt/python@3.11/bin/python3.11

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [6]:
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score


In [7]:
training_path = "Training/train_data.csv"
testing_path = "Testing/test_data.csv"

training_data = pd.read_csv(training_path)
testing_data = pd.read_csv(testing_path)

features = ['num_adults', 'num_kids', 'eating_out_frequency', 'avg_temp_celsius', 'natural_shelf_life_days', 'consumption_rate']
label = 'quantity_consumed'

X_train, y_train = training_data[features], training_data[label]
X_test, y_test = testing_data[features], testing_data[label]

model = XGBRegressor()
model.fit(X_train, y_train)

preds = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, preds))
print('RMSE:', np.sqrt(mean_squared_error(y_test, preds)))
print('R2:', r2_score(y_test, preds))

model.save_model('model.json')



MAE: 0.42478838562965393
RMSE: 0.5379556266165638
R2: 0.9435466527938843


In [31]:
test_sample = testing_data[features].head(20).copy()
test_sample['actual'] = y_test.values[:20]
test_sample['predicted'] = preds[:20].round()
print(test_sample[[ 'num_adults', 'num_kids', 'eating_out_frequency', 'actual', 'predicted']])

    num_adults  num_kids  eating_out_frequency  actual  predicted
0            4         0                 0.037       8        8.0
1            1         0                 0.550       2        1.0
2            2         1                 0.311       2        2.0
3            2         3                 0.197       9       10.0
4            3         1                 0.338       4        4.0
5            1         0                 0.486       2        1.0
6            1         2                 0.499       2        2.0
7            2         0                 0.493       1        2.0
8            1         0                 0.638       2        2.0
9            2         1                 0.578       2        2.0
10           3         0                 0.361       7        7.0
11           1         0                 0.070       1        1.0
12           1         3                 0.132       4        4.0
13           1         0                 0.283       1        1.0
14        

In [23]:
results = testing_data[features].copy()
results['actual'] = y_test.values
results['predicted'] = preds.round()
print(results.head(20))

    num_adults  num_kids  eating_out_frequency  avg_temp_celsius  \
0            4         0                 0.037              18.9   
1            1         0                 0.550              -1.9   
2            2         1                 0.311              -3.8   
3            2         3                 0.197               4.2   
4            3         1                 0.338              -2.1   
5            1         0                 0.486              16.1   
6            1         2                 0.499               0.6   
7            2         0                 0.493              -0.7   
8            1         0                 0.638               4.7   
9            2         1                 0.578              14.3   
10           3         0                 0.361              -1.8   
11           1         0                 0.070               9.5   
12           1         3                 0.132               2.4   
13           1         0                 0.283  

In [26]:
model.load_model('model.json')

new_data = pd.DataFrame([{
    'num_adults': 2,
    'num_kids': 1,
    'eating_out_frequency': 0.4,
    'avg_temp_celsius': 22,
    'natural_shelf_life_days': 7,
    'consumption_rate': 6.0
}])

prediction = model.predict(new_data)[0]
print('Buy: ', round(prediction))

Buy:  7
